# ClimateVariables

Exploracion de variables climaticas disponibles via API para evaluar su utilidad en el proyecto RAIZ.

## Objetivo

Identificar, consultar y documentar variables climaticas utiles para analisis agricola: precipitacion, temperatura, humedad, presion atmosferica y otras variables agroclimaticas disponibles.

Este notebook no entrena modelos. Su proposito es revisar estructura, cobertura temporal, cobertura geografica, unidades, sensores y viabilidad de integracion.

## 1. Configuracion de entorno

En Colab se monta Google Drive para acceder a datos compartidos del equipo cuando sea necesario. Las consultas API deben funcionar sin depender de archivos locales.

Convencion de rutas:

- `DATA_ROOT`: carpeta compartida del equipo, normalmente de solo lectura.
- `PROCESSED_ROOT`: carpeta personal para guardar datasets procesados o salidas temporales.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_ROOT = Path('/content/drive/MyDrive/eco2026')
PROCESSED_ROOT = Path('/content/drive/MyDrive/eco2026_processed')

if IN_COLAB:
    PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

print(f'IN_COLAB={IN_COLAB}')
print(f'DATA_ROOT={DATA_ROOT}')
print(f'PROCESSED_ROOT={PROCESSED_ROOT}')

# Panel de control para ejecuciones seguras con "Run all".
# Activar solo las secciones que se quieran probar en la sesion actual.
EJECUTAR_DESCARGA_PRECIPITACION = False
EJECUTAR_AUDITORIA_PARQUET_PRECIPITACION = False
EJECUTAR_EXPLORACION_TEMPERATURA = False

print('Panel de control cargado. Activa manualmente las banderas necesarias antes de ejecutar secciones costosas.')

## 2. Funciones comunes para APIs

Funciones auxiliares para consultar datasets de datos.gov.co usando Socrata. La idea es reutilizar el mismo patron para cada variable climatica.

In [ ]:
def consultar_datos_gov(dataset_id, select='*', where=None, limit=50000, offset=None, order=None, timeout=60):
    """Consulta un dataset publico de datos.gov.co con una consulta SoQL simple."""
    url = f'https://www.datos.gov.co/resource/{dataset_id}.json'

    query_parts = [f'SELECT {select}']
    if where:
        query_parts.append(f'WHERE {where}')
    if order:
        query_parts.append(f'ORDER BY {order}')
    query_parts.append(f'LIMIT {limit}')
    if offset is not None:
        query_parts.append(f'OFFSET {offset}')

    params = {'$query': ' '.join(query_parts)}
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()

    data = response.json()
    return pd.DataFrame(data)


def resumen_dataframe(df):
    """Resume rapidamente forma, columnas, tipos y nulos de un DataFrame."""
    print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]:,}')
    display(pd.DataFrame({
        'columna': df.columns,
        'tipo': [df[col].dtype for col in df.columns],
        'nulos': [df[col].isna().sum() for col in df.columns],
        'valores_unicos': [df[col].nunique(dropna=True) for col in df.columns],
    }))


def construir_where_in(columna, valores):
    """Construye una condicion SoQL IN escapando comillas simples."""
    valores_limpios = [str(valor).replace("'", "''") for valor in valores]
    valores_sql = ', '.join([f"'{valor}'" for valor in valores_limpios])
    return f'{columna} IN ({valores_sql})'


def valores_distintos(dataset_id, columna, order=None):
    """Lista valores distintos de una columna sin descargar el dataset completo."""
    order = order or columna
    df = consultar_datos_gov(
        dataset_id,
        select=columna,
        where=f'{columna} IS NOT NULL',
        order=order,
        limit=50000
    )
    return df.drop_duplicates().sort_values(columna).reset_index(drop=True)


def total_registros(dataset_id, where=None, timeout=120):
    """Cuenta registros sin descargar el dataset completo."""
    df = consultar_datos_gov(dataset_id, select='count(*) as total_registros', where=where, limit=1, timeout=timeout)
    return int(df.loc[0, 'total_registros'])


def conteo_por_columna(dataset_id, columna, order=None, where_extra=None, timeout=120):
    """Agrupa registros por una columna usando SoQL."""
    order = order or columna
    where_parts = [f'{columna} IS NOT NULL']
    if where_extra:
        where_parts.append(f'({where_extra})')
    where = ' AND '.join(where_parts)
    df = consultar_datos_gov(
        dataset_id,
        select=f'{columna}, count(*) as total_registros',
        limit=50000,
        order=order,
        where=where,
        timeout=timeout
    )
    if 'total_registros' in df.columns:
        df['total_registros'] = pd.to_numeric(df['total_registros'], errors='coerce').astype('Int64')
    return df


def guardar_dataset_procesado(df, filename, subdir='climate_variables', formato=None):
    """Guarda un DataFrame en la carpeta personal de procesados."""
    output_dir = PROCESSED_ROOT / subdir
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / filename
    formato = formato or output_path.suffix.lower().lstrip('.')

    if formato == 'csv':
        df.to_csv(output_path, index=False)
    elif formato == 'parquet':
        df.to_parquet(output_path, index=False)
    else:
        raise ValueError("Formato soportado: 'csv' o 'parquet'")

    print(f'Dataset guardado en: {output_path}')
    return output_path

## 3. Precipitacion

Fuente seleccionada para el proyecto: `s54a-sgyg`.

Decision actual:

- Usar `s54a-sgyg` como fuente principal de precipitacion.
- Acotar las exploraciones a Cundinamarca, Boyaca y Antioquia.
- Descargar por departamento, anio y mes en Parquet particionado.
- No usar `m84s-22dd` como fuente principal; se considera probable duplicado/subconjunto desde 2017.

Riesgo importante: la frecuencia de observacion puede variar por estacion y periodo. Antes de agregar precipitacion a escala diaria, mensual o anual, se debe auditar frecuencia, sensores, duplicados y cobertura.

In [ ]:
PRECIPITACION_FUENTE_PRINCIPAL = 's54a-sgyg'

# Departamentos priorizados para el proyecto RAIZ.
DEPARTAMENTOS_INTERES = [
    'CUNDINAMARCA',
    'BOYACÁ',
    'ANTIOQUIA',
]

print({
    'precipitacion_fuente_principal': PRECIPITACION_FUENTE_PRINCIPAL,
    'departamentos_interes': DEPARTAMENTOS_INTERES,
})

### 3.1 Nota historica de exploracion

Durante la fase de descubrimiento se compararon los datasets `s54a-sgyg` y `m84s-22dd`. En muestras acotadas por departamento, estacion y ventanas cortas, ambos se observaron practicamente equivalentes desde 2017. La diferencia relevante es que `s54a-sgyg` tiene mayor cobertura historica, con fecha minima observada `2003-01-20T15:20:00.000`, mientras `m84s-22dd` inicia en 2017.

Tambien se probaron consultas globales tipo `distinct departamento`, `count(*)` y agrupaciones por departamento. No se mantienen como codigo vivo porque en datasets de millones de filas generaron timeouts o tiempos poco confiables en Socrata. Incluso algunas preguntas acotadas a departamento pueden ser inviables si requieren revisar demasiados registros o calcular agregados pesados desde la API.

Regla de trabajo vigente: usar la API solo para pruebas pequenas y diagnosticos puntuales. Para analisis exploratorios serios, primero descargar los datos de los tres departamentos priorizados a Parquet y luego auditar localmente frecuencia temporal, duplicados, sensores, cobertura y distribucion de valores.

### 3.2 Descarga experimental de precipitacion

Esta seccion prepara una descarga por lotes para el dataset seleccionado (`s54a-sgyg`). La prueba debe empezar con un solo departamento y un solo mes, midiendo tiempo y tamano antes de ampliar la descarga.

La salida se guarda directamente en Parquet particionado dentro de `eco2026_processed`, evitando CSV intermedios.

In [ ]:
PRECIPITACION_FUENTE_PRINCIPAL = 's54a-sgyg'

# Seguridad: esta bandera se define en el panel de control de la seccion 1.
EJECUTAR_DESCARGA_PRECIPITACION = globals().get('EJECUTAR_DESCARGA_PRECIPITACION', False)

# Modo de descarga:
# - 'mes': descarga una sola particion departamento/anio/mes.
# - 'plan': recorre listas de departamentos, anios y meses.
DESCARGA_MODO = 'mes'

# Prueba inicial recomendada: un departamento y un mes.
DESCARGA_DEPARTAMENTO = 'ANTIOQUIA'
DESCARGA_ANIO = 2026
DESCARGA_MES = 1

# Para DESCARGA_MODO = 'plan'.
# Ejemplos:
# - Un departamento durante 2026: DESCARGA_DEPARTAMENTOS = ['ANTIOQUIA']; DESCARGA_ANIOS = [2026]
# - Tres departamentos durante 2026: DESCARGA_DEPARTAMENTOS = DEPARTAMENTOS_INTERES; DESCARGA_ANIOS = [2026]
# - Rango de anios: DESCARGA_ANIOS = list(range(2003, 2006))
DESCARGA_DEPARTAMENTOS = [DESCARGA_DEPARTAMENTO]
DESCARGA_ANIOS = [DESCARGA_ANIO]
DESCARGA_MESES = list(range(1, 13))

# Socrata pagina con LIMIT + OFFSET dentro del $query.
# Usamos 1000 como valor conservador porque muchos endpoints publicos no entregan mas por consulta.
DESCARGA_LIMIT = 1000
DESCARGA_MAX_LOTES = 10  # En modo plan aplica por cada mes. Cambiar a None para completar cada mes.
SOBRESCRIBIR_PARQUET = False

print({
    'dataset': PRECIPITACION_FUENTE_PRINCIPAL,
    'modo': DESCARGA_MODO,
    'departamento': DESCARGA_DEPARTAMENTO,
    'anio': DESCARGA_ANIO,
    'mes': DESCARGA_MES,
    'departamentos_plan': DESCARGA_DEPARTAMENTOS,
    'anios_plan': DESCARGA_ANIOS,
    'meses_plan': DESCARGA_MESES,
    'limit': DESCARGA_LIMIT,
    'max_lotes': DESCARGA_MAX_LOTES,
})

In [ ]:
def inicio_mes_siguiente(anio, mes):
    """Devuelve el primer dia del mes siguiente."""
    if mes == 12:
        return anio + 1, 1
    return anio, mes + 1


def normalizar_precipitacion_cruda(df, dataset_id):
    """Aplica conversiones minimas antes de guardar el lote crudo."""
    df = df.copy()
    df['dataset_id'] = dataset_id

    if 'fechaobservacion' in df.columns:
        df['fechaobservacion'] = pd.to_datetime(df['fechaobservacion'], errors='coerce')
    if 'valorobservado' in df.columns:
        df['valorobservado'] = pd.to_numeric(df['valorobservado'], errors='coerce')
    for columna in ['latitud', 'longitud']:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors='coerce')

    return df


def ruta_precipitacion_parquet(dataset_id, departamento, anio, mes):
    """Construye la carpeta particionada para un mes de precipitacion."""
    departamento_particion = str(departamento).replace(' ', '_')
    return (
        PROCESSED_ROOT
        / 'precipitacion'
        / f'fuente={dataset_id}'
        / f'departamento={departamento_particion}'
        / f'anio={anio}'
        / f'mes={mes:02d}'
    )


def descargar_precipitacion_mes(
    dataset_id,
    departamento,
    anio,
    mes,
    limit=50000,
    max_lotes=1,
    sobrescribir=False,
    timeout=180,
):
    """Descarga un mes por lotes y guarda cada lote como Parquet."""
    try:
        import pyarrow  # noqa: F401
    except ImportError as exc:
        raise ImportError('Para guardar Parquet en Colab instala pyarrow: !pip install pyarrow') from exc

    import time
    from datetime import datetime

    anio_fin, mes_fin = inicio_mes_siguiente(anio, mes)
    fecha_inicio = f'{anio:04d}-{mes:02d}-01T00:00:00'
    fecha_fin = f'{anio_fin:04d}-{mes_fin:02d}-01T00:00:00'
    departamento_sql = str(departamento).replace("'", "''")

    where = (
        f"departamento = '{departamento_sql}' "
        f"AND fechaobservacion >= '{fecha_inicio}' "
        f"AND fechaobservacion < '{fecha_fin}'"
    )

    output_dir = ruta_precipitacion_parquet(dataset_id, departamento, anio, mes)
    output_dir.mkdir(parents=True, exist_ok=True)

    total_filas = 0
    lotes = []
    offset = 0
    lote_idx = 0
    lotes_consultados = 0
    inicio = time.perf_counter()
    inicio_reloj = datetime.now()
    print(f'Inicio descarga: {inicio_reloj:%Y-%m-%d %H:%M:%S}')

    while True:
        if max_lotes is not None and lotes_consultados >= max_lotes:
            print(f'Corte por max_lotes={max_lotes}.')
            break

        lote_inicio = time.perf_counter()
        lote_inicio_reloj = datetime.now()
        output_path = output_dir / f'part-{lote_idx:05d}.parquet'

        if output_path.exists() and not sobrescribir:
            lote_fin_reloj = datetime.now()
            duracion_lote = time.perf_counter() - lote_inicio
            print(f'Ya existe {output_path.name}; se omite consulta y se continua con el siguiente lote.')
            lotes.append({
                'lote': lote_idx,
                'offset': offset,
                'filas': 0,
                'estado': 'omitido_existente',
                'duracion_segundos': round(duracion_lote, 2),
                'inicio': lote_inicio_reloj.isoformat(timespec='seconds'),
                'fin': lote_fin_reloj.isoformat(timespec='seconds'),
                'archivo': str(output_path),
            })
            lote_idx += 1
            offset += limit
            continue

        print(f'Consultando lote {lote_idx} | offset={offset:,} | limit={limit:,} | inicio={lote_inicio_reloj:%H:%M:%S}')
        df_lote = consultar_datos_gov(
            dataset_id,
            where=where,
            limit=limit,
            offset=offset,
            order='fechaobservacion, codigoestacion',
            timeout=timeout,
        )

        if df_lote.empty:
            print('La API no devolvio mas filas.')
            break

        lotes_consultados += 1
        df_lote = normalizar_precipitacion_cruda(df_lote, dataset_id)
        estado_lote = 'sobrescrito' if output_path.exists() else 'escrito'
        df_lote.to_parquet(output_path, index=False)

        filas_lote = len(df_lote)
        lote_fin_reloj = datetime.now()
        duracion_lote = time.perf_counter() - lote_inicio
        total_filas += filas_lote
        lotes.append({
            'lote': lote_idx,
            'offset': offset,
            'filas': filas_lote,
            'estado': estado_lote,
            'duracion_segundos': round(duracion_lote, 2),
            'inicio': lote_inicio_reloj.isoformat(timespec='seconds'),
            'fin': lote_fin_reloj.isoformat(timespec='seconds'),
            'archivo': str(output_path),
        })

        if filas_lote < limit:
            print('Ultimo lote detectado: filas_lote < limit.')
            break

        lote_idx += 1
        offset += limit

    fin_reloj = datetime.now()
    duracion = time.perf_counter() - inicio
    resumen = pd.DataFrame(lotes)
    print(f'Fin descarga: {fin_reloj:%Y-%m-%d %H:%M:%S}')
    print(f'Filas consultadas en esta corrida: {total_filas:,}')
    print(f'Duracion: {duracion:.1f} segundos')
    print(f'Carpeta salida: {output_dir}')
    return resumen


def construir_plan_descarga_precipitacion(
    modo,
    departamento,
    anio,
    mes,
    departamentos,
    anios,
    meses,
):
    """Construye las particiones mensuales a descargar."""
    if modo == 'mes':
        return [{'departamento': departamento, 'anio': anio, 'mes': mes}]

    if modo != 'plan':
        raise ValueError("DESCARGA_MODO debe ser 'mes' o 'plan'")

    return [
        {'departamento': depto, 'anio': anio_item, 'mes': mes_item}
        for depto in departamentos
        for anio_item in anios
        for mes_item in meses
    ]


def descargar_precipitacion_plan(
    dataset_id,
    plan,
    limit=1000,
    max_lotes_por_mes=1,
    sobrescribir=False,
):
    """Ejecuta descargas mensuales segun un plan de particiones."""
    resumenes = []

    for idx, item in enumerate(plan, start=1):
        print('\n' + '=' * 80)
        print(
            f"Particion {idx:,}/{len(plan):,}: "
            f"{item['departamento']} | {item['anio']}-{item['mes']:02d}"
        )
        resumen_mes = descargar_precipitacion_mes(
            dataset_id=dataset_id,
            departamento=item['departamento'],
            anio=item['anio'],
            mes=item['mes'],
            limit=limit,
            max_lotes=max_lotes_por_mes,
            sobrescribir=sobrescribir,
        )

        if not resumen_mes.empty:
            resumen_mes = resumen_mes.copy()
            resumen_mes['departamento'] = item['departamento']
            resumen_mes['anio'] = item['anio']
            resumen_mes['mes'] = item['mes']
            resumenes.append(resumen_mes)

    if not resumenes:
        return pd.DataFrame()

    return pd.concat(resumenes, ignore_index=True)


In [ ]:
if EJECUTAR_DESCARGA_PRECIPITACION:
    plan_descarga_precipitacion = construir_plan_descarga_precipitacion(
        modo=DESCARGA_MODO,
        departamento=DESCARGA_DEPARTAMENTO,
        anio=DESCARGA_ANIO,
        mes=DESCARGA_MES,
        departamentos=DESCARGA_DEPARTAMENTOS,
        anios=DESCARGA_ANIOS,
        meses=DESCARGA_MESES,
    )
    print(f'Particiones a procesar: {len(plan_descarga_precipitacion):,}')
    resumen_descarga_precipitacion = descargar_precipitacion_plan(
        dataset_id=PRECIPITACION_FUENTE_PRINCIPAL,
        plan=plan_descarga_precipitacion,
        limit=DESCARGA_LIMIT,
        max_lotes_por_mes=DESCARGA_MAX_LOTES,
        sobrescribir=SOBRESCRIBIR_PARQUET,
    )
    display(resumen_descarga_precipitacion)
else:
    print('Descarga experimental desactivada. Cambiar EJECUTAR_DESCARGA_PRECIPITACION=True para probar en Colab.')

### 3.3 Auditoria inicial desde Parquet

Una vez existan archivos Parquet descargados, las exploraciones costosas deben hacerse sobre esos archivos y no contra la API. Esta auditoria revisa rapidamente filas, rango temporal, estaciones, duplicados y frecuencia entre observaciones para una particion mensual.

In [ ]:
AUDITORIA_DEPARTAMENTO = DESCARGA_DEPARTAMENTO
AUDITORIA_ANIO = DESCARGA_ANIO
AUDITORIA_MES = DESCARGA_MES

print({
    'departamento': AUDITORIA_DEPARTAMENTO,
    'anio': AUDITORIA_ANIO,
    'mes': AUDITORIA_MES,
})

In [ ]:
def listar_parquets_precipitacion(dataset_id, departamento, anio, mes):
    """Lista archivos Parquet disponibles para una particion mensual."""
    carpeta = ruta_precipitacion_parquet(dataset_id, departamento, anio, mes)
    archivos = sorted(carpeta.glob('part-*.parquet'))
    return carpeta, archivos


def leer_precipitacion_parquet(dataset_id, departamento, anio, mes):
    """Lee todos los Parquet de una particion mensual."""
    carpeta, archivos = listar_parquets_precipitacion(dataset_id, departamento, anio, mes)
    if not archivos:
        print(f'No hay archivos Parquet en {carpeta}')
        return pd.DataFrame()

    dataframes = [pd.read_parquet(archivo) for archivo in archivos]
    df = pd.concat(dataframes, ignore_index=True)
    if 'fechaobservacion' in df.columns:
        df['fechaobservacion'] = pd.to_datetime(df['fechaobservacion'], errors='coerce')
    return df


def auditar_precipitacion_parquet(df):
    """Resume calidad basica y frecuencia temporal por estacion/sensor."""
    if df.empty:
        print('Sin datos para auditar.')
        return pd.DataFrame(), pd.DataFrame()

    print(f'Filas: {len(df):,}')
    print(f"Fecha minima: {df['fechaobservacion'].min()}")
    print(f"Fecha maxima: {df['fechaobservacion'].max()}")
    print(f"Estaciones: {df['codigoestacion'].nunique():,}")

    claves_duplicado = ['codigoestacion', 'codigosensor', 'fechaobservacion']
    claves_presentes = [col for col in claves_duplicado if col in df.columns]
    duplicados = df.duplicated(subset=claves_presentes).sum() if claves_presentes else 0
    print(f'Duplicados por {claves_presentes}: {duplicados:,}')

    columnas_frecuencia = ['codigoestacion', 'codigosensor']
    columnas_frecuencia = [col for col in columnas_frecuencia if col in df.columns]
    df_ordenado = df.sort_values(columnas_frecuencia + ['fechaobservacion']).copy()
    df_ordenado['minutos_desde_anterior'] = (
        df_ordenado.groupby(columnas_frecuencia)['fechaobservacion']
        .diff()
        .dt.total_seconds()
        .div(60)
    )

    frecuencia = (
        df_ordenado
        .dropna(subset=['minutos_desde_anterior'])
        .groupby(columnas_frecuencia)['minutos_desde_anterior']
        .agg(['count', 'median', 'min', 'max'])
        .reset_index()
        .sort_values(['median', 'count'], ascending=[True, False])
    )

    estaciones = (
        df.groupby(['codigoestacion', 'departamento', 'municipio'], dropna=False)
        .agg(
            filas=('fechaobservacion', 'size'),
            fecha_min=('fechaobservacion', 'min'),
            fecha_max=('fechaobservacion', 'max'),
            sensores=('codigosensor', 'nunique'),
        )
        .reset_index()
        .sort_values('filas', ascending=False)
    )

    return estaciones, frecuencia


In [ ]:
if EJECUTAR_AUDITORIA_PARQUET_PRECIPITACION:
    carpeta_auditoria, archivos_auditoria = listar_parquets_precipitacion(
        PRECIPITACION_FUENTE_PRINCIPAL,
        AUDITORIA_DEPARTAMENTO,
        AUDITORIA_ANIO,
        AUDITORIA_MES,
    )
    print(f'Carpeta: {carpeta_auditoria}')
    print(f'Archivos Parquet: {len(archivos_auditoria):,}')

    df_precipitacion_parquet = leer_precipitacion_parquet(
        PRECIPITACION_FUENTE_PRINCIPAL,
        AUDITORIA_DEPARTAMENTO,
        AUDITORIA_ANIO,
        AUDITORIA_MES,
    )
    estaciones_precipitacion, frecuencia_precipitacion = auditar_precipitacion_parquet(df_precipitacion_parquet)
    display(estaciones_precipitacion.head(20))
    display(frecuencia_precipitacion.head(20))
else:
    print('Auditoria de Parquet desactivada. Cambiar EJECUTAR_AUDITORIA_PARQUET_PRECIPITACION=True para revisar archivos descargados.')

## 4. Temperatura

Exploracion de temperatura ambiente del aire. En exploraciones previas aparece el dataset `sbwg-7ju4`, pero conviene validar cobertura, columnas y formato de fecha antes de usarlo como fuente definitiva.

In [ ]:
TEMPERATURA_DATASET_ID = 'sbwg-7ju4'

if EJECUTAR_EXPLORACION_TEMPERATURA:
    df_temperatura = consultar_datos_gov(TEMPERATURA_DATASET_ID, limit=1000)
    display(df_temperatura.head())
    resumen_dataframe(df_temperatura)
else:
    print('Exploracion de temperatura desactivada.')

## 5. Humedad

Registrar aqui datasets candidatos de humedad relativa, humedad del suelo u otras variables asociadas.

In [ ]:
# TODO: completar con dataset_id de humedad.
HUMEDAD_DATASET_ID = ''

if HUMEDAD_DATASET_ID:
    df_humedad = consultar_datos_gov(HUMEDAD_DATASET_ID, limit=1000)
    display(df_humedad.head())
    resumen_dataframe(df_humedad)
else:
    print('Pendiente: definir HUMEDAD_DATASET_ID')

## 6. Presion atmosferica

Registrar aqui el dataset de presion atmosferica y validar si aporta informacion util para rendimiento agricola o si se conserva solo como variable complementaria.

In [ ]:
# TODO: completar con dataset_id de presion atmosferica.
PRESION_DATASET_ID = ''

if PRESION_DATASET_ID:
    df_presion = consultar_datos_gov(PRESION_DATASET_ID, limit=1000)
    display(df_presion.head())
    resumen_dataframe(df_presion)
else:
    print('Pendiente: definir PRESION_DATASET_ID')

## 7. Otras variables agroclimaticas

Espacio para explorar variables adicionales: radiacion solar, brillo solar, velocidad del viento, eventos extremos, indices Niño/Niña u otras fuentes relevantes.

In [ ]:
variables_adicionales = []

# Ejemplo de registro manual:
# variables_adicionales.append({
#     'variable': 'radiacion_solar',
#     'dataset_id': 'pendiente',
#     'fuente': 'datos.gov.co',
#     'estado': 'por revisar',
# })

pd.DataFrame(variables_adicionales)

## 8. Tabla resumen de variables

Tabla de decision para comparar rapidamente las variables encontradas.

In [ ]:
resumen_variables = pd.DataFrame(columns=[
    'variable',
    'dataset_id',
    'fuente',
    'unidad',
    'frecuencia_observada',
    'nivel_geografico',
    'fecha_min',
    'fecha_max',
    'columnas_clave',
    'riesgos',
    'decision'
])

resumen_variables

## 9. Conclusiones preliminares

Anotar aqui hallazgos, dudas y decisiones. La salida esperada no es un modelo, sino una recomendacion sobre que variables climaticas pasan a la siguiente fase de integracion.